# `share_dispersion` Calibration — Projection 2050 — Norte Amazónica

**Context:** the 3 pass-1 2050 case studies (`no_transition`, `late_access`, `early_access` --
`early_access_brazil` was never run, per instruction) all solved with `share_dispersion = 0`
everywhere (its pass-1 default), same starting point as 2035's own pass 1. Under that default,
`share_tech_hs` (`ESMC_model_AMPL.mod:642-644`) is non-binding, so nothing forces `TECH_HS`
(`PV_HS`+`HS_DIESEL` net) to actually serve the dispersed households it is meant to represent --
confirmed in this session's pass-1 reports: C2-C4 overshoot (`PV_HS`+`HS_DIESEL` production landing
at or near 2x the target in `no_transition`, exactly on target but 100% `HS_DIESEL`-sourced in
`late_access`/`early_access`), and C1 lands at exactly 0/2.079 GWh in all three -- served entirely
via SIN import instead, which is physically impossible for households no line reaches.

**Method (identical to the 2035 pass-3 calibration, `2035/technologies.ipynb` Section 9):**

```
share_dispersion[c] = target[c] / (target[c] + offre_locale_hors_TECH_HS[c])
```

`offre_locale_hors_TECH_HS[c]` = production of every technology feeding the `ELECTRICITY` layer in
that scenario's own pass-1 solve (`Year_balance.csv`, excluding the `ELECTRICITY` balance row
itself and excluding `PV_HS`/`HS_DIESEL`/`BATT_HS` -- the `TECH_HS` side), **plus**
`R_year_local` + `R_year_exterior` for `ELECTRICITY` from that region's own `Resources.csv`.
`R_year_import`/`R_year_export` (inter-cluster exchange) are deliberately excluded --
`Year_balance.csv`'s `ELECTRICITY` row nets exterior import together with inter-cluster exchange
into one aggregate flow, so `Resources.csv` is read directly instead, per region, to keep the two
apart. 2050 dispersed-demand targets (`cluster_summary.csv`): C1 2.079, C2 0.132, C3 0.648,
C4 1.543, C5 0.000 GWh/y.

**C2 and C5 forced to 0** after the formula, same as the 2035 pass-3 correction: C2 imports nearly
all its electricity from C5 (denominator too close to zero for `share_dispersion` to reliably
bite there; the live mechanism for C2 is `f_max_prod`'s cap alone, uniform "cap-only" regime, no
floor); C5's target is 0, so its formula value is already 0 -- forced explicitly here for clarity,
not because the raw computation would differ.

**`early_access_brazil`** has no 2050 pass-1 solve of its own (never run) -- it mirrors
`early_access`'s calibrated value, same lineage already used throughout this pipeline for its
chained `f_min`, `Demands.csv`, and `DIESEL.avail_exterior`.


In [1]:
import pandas as pd
import json

ROOT = "../../../EnergyScope_BO_nord_amazonia"
CS_ROOT_2050 = f"{ROOT}/case_studies/C1_C2_C3_C4_C5"
CALIBRATION_SCENARIOS_2050 = ["no_transition", "late_access", "early_access"]
DISPERSED_DEMAND_GWH_BY_CLUSTER = {1: 2.079, 2: 0.132, 3: 0.648, 4: 1.543, 5: 0.000}

EXCLUDE_TECHS_FOR_OFFRE_LOCALE = {"ELECTRICITY", "PV_HS", "HS_DIESEL", "BATT_HS"}

SHARE_DISPERSION_BY_SCENARIO_2050 = {}   # scenario -> {k: raw share_dispersion, before C2/C5 forcing}
OFFRE_LOCALE_BY_SCENARIO_2050 = {}       # scenario -> {k: offre_locale_hors_TECH_HS}, kept for the record

for scenario in CALIBRATION_SCENARIOS_2050:
    cs_outputs = f"{CS_ROOT_2050}/norte_amazonia_{scenario}_2050/outputs"

    # Always verify the pass-1 source actually solved before trusting its output -- the pipeline
    # writes normal-looking files even on a rejected solve.
    solve_info = pd.read_csv(f"{cs_outputs}/Solve_info.csv", sep=r"\t;\t", header=None,
                              index_col=0, engine="python")
    solve_result_num = int(float(solve_info.loc["solve_result_num", 1]))
    assert solve_result_num == 0, (
        f"{scenario}_2050: solve_result_num={solve_result_num} != 0 -- "
        f"refusing to calibrate share_dispersion from a non-optimal pass-1 solve")

    cs_dir = f"{cs_outputs}/regional_results"
    yb = pd.read_csv(f"{cs_dir}/Year_balance.csv", sep=";")
    yb["ELECTRICITY"] = pd.to_numeric(yb["ELECTRICITY"], errors="coerce")
    yb_tech = yb[~yb["Elements"].isin(EXCLUDE_TECHS_FOR_OFFRE_LOCALE)]
    yb_tech = yb_tech[yb_tech["ELECTRICITY"] > 1e-9]
    tech_sum = yb_tech.groupby("Regions")["ELECTRICITY"].sum()

    res = pd.read_csv(f"{cs_dir}/Resources.csv", sep=";")
    res_elec = res[res["Resources"] == "ELECTRICITY"].set_index("Regions")
    r_local = pd.to_numeric(res_elec["R_year_local"], errors="coerce").fillna(0.0)
    r_ext = pd.to_numeric(res_elec["R_year_exterior"], errors="coerce").fillna(0.0)
    # R_year_import / R_year_export deliberately excluded -- inter-cluster exchange, not local supply

    shares, offres = {}, {}
    for k in range(1, 6):
        region = f"C{k}"
        offre_locale = (float(tech_sum.get(region, 0.0))
                         + float(r_local.get(region, 0.0)) + float(r_ext.get(region, 0.0)))
        target = DISPERSED_DEMAND_GWH_BY_CLUSTER[k]
        share = target / (target + offre_locale) if (target + offre_locale) > 0 else 0.0
        shares[k] = share
        offres[k] = offre_locale

    SHARE_DISPERSION_BY_SCENARIO_2050[scenario] = shares
    OFFRE_LOCALE_BY_SCENARIO_2050[scenario] = offres
    print(f"{scenario}_2050 (solve_result_num={solve_result_num}):")
    print(f"  offre_locale_hors_TECH_HS = {offres}")
    print(f"  share_dispersion (raw)    = {shares}")


no_transition_2050 (solve_result_num=0):
  offre_locale_hors_TECH_HS = {1: 38.97724665794309, 2: 0.1454185505656317, 3: 166.01756104283524, 4: 86.83360526177559, 5: 82.55914989732281}
  share_dispersion (raw)    = {1: 0.05063784854278147, 2: 0.47581533293596906, 3: 0.0038880257921638382, 4: 0.017459371690387547, 5: 0.0}
late_access_2050 (solve_result_num=0):
  offre_locale_hors_TECH_HS = {1: 38.97724665794309, 2: 3.625372729783464, 3: 203.01837394126449, 4: 117.54300677144936, 5: 99.59240010854688}
  share_dispersion (raw)    = {1: 0.05063784854278147, 2: 0.035130930438090215, 3: 0.003181673967381956, 4: 0.0129570219191356, 5: 0.0}
early_access_2050 (solve_result_num=0):
  offre_locale_hors_TECH_HS = {1: 38.97676389574444, 2: 3.914559203917283, 3: 203.0273817634152, 4: 117.6203262513943, 5: 99.23543271527822}
  share_dispersion (raw)    = {1: 0.05063844397778932, 2: 0.032620306128776526, 3: 0.0031815332535019003, 4: 0.012948614716786202, 5: 0.0}


## Force C2 and C5 to 0, mirror `early_access_brazil` from `early_access`

In [2]:
DEPLOYED_SHARE_DISPERSION_2050 = {}
for scenario in CALIBRATION_SCENARIOS_2050:
    deployed = dict(SHARE_DISPERSION_BY_SCENARIO_2050[scenario])
    deployed[2] = 0.0  # C2 forced to 0 -- uniform cap-only regime, same as the 2035 pass-3 correction
    deployed[5] = 0.0  # C5 target is 0 -- already 0 by the formula, forced explicitly for clarity
    DEPLOYED_SHARE_DISPERSION_2050[scenario] = deployed
    print(f"{scenario}_2050 deployed share_dispersion: {deployed}")

# early_access_brazil mirrors early_access -- no 2050 pass-1 solve of its own.
DEPLOYED_SHARE_DISPERSION_2050["early_access_brazil"] = dict(DEPLOYED_SHARE_DISPERSION_2050["early_access"])
print(f"early_access_brazil_2050 deployed share_dispersion (mirrors early_access): "
      f"{DEPLOYED_SHARE_DISPERSION_2050['early_access_brazil']}")

DEPLOY_SCENARIOS_2050 = ["no_transition", "late_access", "early_access", "early_access_brazil"]


no_transition_2050 deployed share_dispersion: {1: 0.05063784854278147, 2: 0.0, 3: 0.0038880257921638382, 4: 0.017459371690387547, 5: 0.0}
late_access_2050 deployed share_dispersion: {1: 0.05063784854278147, 2: 0.0, 3: 0.003181673967381956, 4: 0.0129570219191356, 5: 0.0}
early_access_2050 deployed share_dispersion: {1: 0.05063844397778932, 2: 0.0, 3: 0.0031815332535019003, 4: 0.012948614716786202, 5: 0.0}
early_access_brazil_2050 deployed share_dispersion (mirrors early_access): {1: 0.05063844397778932, 2: 0.0, 3: 0.0031815332535019003, 4: 0.012948614716786202, 5: 0.0}


## Deploy to `Misc.json`, with assertion

In [3]:
for scenario in DEPLOY_SCENARIOS_2050:
    for k in range(1, 6):
        target = f"{ROOT}/Data/2050/{scenario}/C{k}/Misc.json"
        deployed_value = DEPLOYED_SHARE_DISPERSION_2050[scenario][k]

        with open(target, encoding="utf-8") as f:
            misc = json.load(f)
        misc["share_dispersion"] = deployed_value
        with open(target, "w", encoding="utf-8") as f:
            json.dump(misc, f, indent=4)

        with open(target, encoding="utf-8") as f:
            check = json.load(f)
        actual = check["share_dispersion"]
        if abs(actual - deployed_value) > 1e-9:
            raise AssertionError(f"{target}: share_dispersion = {actual}, expected {deployed_value}")
    print(f"OK -- share_dispersion deployed and verified in Misc.json (C1-C5) for 2050/{scenario}")


OK -- share_dispersion deployed and verified in Misc.json (C1-C5) for 2050/no_transition
OK -- share_dispersion deployed and verified in Misc.json (C1-C5) for 2050/late_access
OK -- share_dispersion deployed and verified in Misc.json (C1-C5) for 2050/early_access
OK -- share_dispersion deployed and verified in Misc.json (C1-C5) for 2050/early_access_brazil


## Reprint `reg_misc.dat` from the deployed catalog (no solve)

Same mechanism used for the GENSET_DIESEL and DIESEL.avail_exterior fixes earlier this session:
instantiate `Esmc`, read the just-deployed `Data/2050/{scenario}/` catalog, apply the same
pre-solve preprocessing `scripts/run.py` applies, reuse the shared typical-day cache
(`algo='read'`, no AMPL call), then `print_data(indep=True)`. **No `set_esom()`/`solve_esom()`
call -- no AMPL solve is launched.**


In [4]:
import sys
sys.path.insert(0, r"C:\Valen\Tfe\EnergyScope_BO_nord_amazonia")
from pathlib import Path
from esmc import Esmc

REPRINT_YEAR = 2050
REPRINT_CASE_STUDY = {s: f"norte_amazonia_{s}_{REPRINT_YEAR}" for s in DEPLOY_SCENARIOS_2050}
FT_TO_DROP = ['BIOMASS_TO_GASOLINE', 'BIOMASS_TO_DIESEL', 'BIOWASTE_TO_GASOLINE', 'BIOWASTE_TO_DIESEL',
              'POWER_TO_GASOLINE', 'POWER_TO_DIESEL', 'H2_TO_GASOLINE', 'H2_TO_DIESEL']
AMPL_PATH = r'C:\Users\valen\AMPL'  # unused by algo='read'

REPRINT_MODELS = {}
for scenario in DEPLOY_SCENARIOS_2050:
    case_study = REPRINT_CASE_STUDY[scenario]
    config = {'case_study': case_study, 'comment': 'share_dispersion reprint (no solve)',
              'regions_names': ['C1', 'C2', 'C3', 'C4', 'C5'],
              'gwp_limit_overall': None, 're_share_primary': None, 'f_perc': True,
              'year': REPRINT_YEAR, 'scenario': scenario}

    my_model = Esmc(config, nbr_td=16)
    current_project = Path(r"C:\Valen\Tfe\EnergyScope_BO_nord_amazonia")
    my_model.project_dir = current_project
    my_model.dat_dir = current_project / 'case_studies' / my_model.space_id / '00_td_dat'
    my_model.cs_dir = current_project / 'case_studies' / my_model.space_id / case_study
    my_model.dat_dir.mkdir(parents=True, exist_ok=True)
    my_model.cs_dir.mkdir(parents=True, exist_ok=True)

    my_model.read_data_indep()
    my_model.init_regions()

    my_model.ref_region.data['Technologies'] = my_model.ref_region.data['Technologies'].drop(index=FT_TO_DROP)
    my_model.data_indep['Layers_in_out'] = my_model.data_indep['Layers_in_out'].drop(index=FT_TO_DROP)
    for r_code, region in my_model.regions.items():
        region.data['Technologies'] = region.data['Technologies'].drop(index=FT_TO_DROP)

    if case_study.startswith('norte_amazonia_early_access_') or case_study.startswith('norte_amazonia_late_access_'):
        for r_code, region in my_model.regions.items():
            region.data['Technologies'].loc['PV_UTILITY', 'f_max'] = 1e15
            region.data['Technologies'].loc['BATT_LI', 'f_max'] = 1e15

    # Pre-print sanity check: in-memory Misc data (freshly read from the deployed Misc.json files)
    # must match the just-deployed share_dispersion, before print_data() writes anything.
    for k in range(1, 6):
        region_code = f"C{k}"
        in_memory_sd = float(my_model.regions[region_code].data['Misc']['share_dispersion'])
        expected = DEPLOYED_SHARE_DISPERSION_2050[scenario][k]
        assert abs(in_memory_sd - expected) < 1e-9, (
            f"{scenario} {region_code}: in-memory share_dispersion={in_memory_sd} after "
            f"init_regions(), expected {expected} (deployed Misc.json)")

    my_model.init_ta(algo='read', ampl_path=AMPL_PATH)
    my_model.print_td_data()
    my_model.print_data(indep=True)

    REPRINT_MODELS[scenario] = my_model
    print(f"OK -- reg_misc.dat reprinted for {case_study} at {my_model.cs_dir} "
          f"(pre-print in-memory check passed, no solve)")


[INFO    ] (read_data_indep): Read indep data from C:\Valen\Tfe\EnergyScope_BO_nord_amazonia\Data\2050\no_transition\00_INDEP


[INFO    ] (init_regions): Initialising regions: C1, C2, C3, C4, C5


[INFO    ] (read_data): Read data from C:\Valen\Tfe\EnergyScope_BO_nord_amazonia\Data\2050\no_transition\02_REF_REGION


[INFO    ] (read_data): Read data from C:\Valen\Tfe\EnergyScope_BO_nord_amazonia\Data\2050\no_transition\C1


[INFO    ] (read_data): Read data from C:\Valen\Tfe\EnergyScope_BO_nord_amazonia\Data\2050\no_transition\C2


...

[INFO    ] (generate_t_h_td): t_h_td and td_count generated


[INFO    ] (print_td_data): Printing TD data into C:\Valen\Tfe\EnergyScope_BO_nord_amazonia\case_studies\C1_C2_C3_C4_C5\norte_amazonia_early_access_brazil_2050\reg_16TD.dat


[INFO    ] (generate_t_h_td): t_h_td and td_count generated


[INFO    ] (print_data): Printing regional data into C:\Valen\Tfe\EnergyScope_BO_nord_amazonia\case_studies\C1_C2_C3_C4_C5\norte_amazonia_early_access_brazil_2050


OK -- reg_misc.dat reprinted for norte_amazonia_early_access_brazil_2050 at C:\Valen\Tfe\EnergyScope_BO_nord_amazonia\case_studies\C1_C2_C3_C4_C5\norte_amazonia_early_access_brazil_2050 (pre-print in-memory check passed, no solve)


### Spot-check the reprinted `.dat` text itself

In [5]:
print("=== reg_misc.dat text spot-check (parses the actual share_dispersion value) ===")
for scenario in DEPLOY_SCENARIOS_2050:
    dat_path = REPRINT_MODELS[scenario].cs_dir / "reg_misc.dat"
    text = dat_path.read_text(encoding="utf-8")
    lines = text.splitlines()
    # share_ned is printed first as "param share_ned :" -- share_dispersion is printed later in a
    # generic "param :" block; find the block whose header includes share_dispersion.
    header_line_idx = next(i for i, l in enumerate(lines)
                            if l.strip().startswith("param") and "share_dispersion" in l)
    header_fields = lines[header_line_idx].split()
    # header_fields = ['param', ':', 'col_a', 'col_b', 'share_dispersion', ':=']; data rows are
    # "REGION  val_a  val_b  val_share_dispersion" -- named column i (0-based, after 'param' ':')
    # lands at row position 1+i, and header_fields.index(name) = 2+i, so row position = index-1.
    col_idx = header_fields.index("share_dispersion") - 1
    for k in range(1, 6):
        row = next((l for l in lines[header_line_idx+1:] if l.split() and l.split()[0] == f"C{k}"), None)
        assert row is not None, f"{scenario} reg_misc.dat: no C{k} row found in the share_dispersion block"
        printed_sd = float(row.split()[col_idx])
        expected = DEPLOYED_SHARE_DISPERSION_2050[scenario][k]
        assert abs(printed_sd - expected) < 1e-6, (
            f"{scenario} reg_misc.dat C{k}: printed share_dispersion={printed_sd}, expected {expected}")
    print(f"OK -- {dat_path}: share_dispersion matches the deployed catalog in all 5 clusters")

print()
print("Reprint complete for all 4 scenarios. No set_esom()/solve_esom() call was made -- "
      "no AMPL solve was launched.")


=== reg_misc.dat text spot-check (parses the actual share_dispersion value) ===
OK -- C:\Valen\Tfe\EnergyScope_BO_nord_amazonia\case_studies\C1_C2_C3_C4_C5\norte_amazonia_no_transition_2050\reg_misc.dat: share_dispersion matches the deployed catalog in all 5 clusters
OK -- C:\Valen\Tfe\EnergyScope_BO_nord_amazonia\case_studies\C1_C2_C3_C4_C5\norte_amazonia_late_access_2050\reg_misc.dat: share_dispersion matches the deployed catalog in all 5 clusters
OK -- C:\Valen\Tfe\EnergyScope_BO_nord_amazonia\case_studies\C1_C2_C3_C4_C5\norte_amazonia_early_access_2050\reg_misc.dat: share_dispersion matches the deployed catalog in all 5 clusters
OK -- C:\Valen\Tfe\EnergyScope_BO_nord_amazonia\case_studies\C1_C2_C3_C4_C5\norte_amazonia_early_access_brazil_2050\reg_misc.dat: share_dispersion matches the deployed catalog in all 5 clusters

Reprint complete for all 4 scenarios. No set_esom()/solve_esom() call was made -- no AMPL solve was launched.
